# 🔧 AWQ Quantization: Qwen3-4B-Instruct

This notebook walks you through quantizing **Qwen3-4B-Instruct** from FP16 to AWQ INT4 format.

**What you'll learn:**
- How to configure AWQ quantization parameters
- How to fetch and use calibration data
- How to run the quantization process
- How to verify the quantized model

**Requirements:**
- NVIDIA GPU with ≥16 GB VRAM
- Python 3.10+, PyTorch 2.4+, AutoAWQ 0.2.7+

**Time:** ~10-15 minutes on H100

## 1. Install Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install autoawq>=0.2.7 transformers>=4.51.0 accelerate datasets torch>=2.4.0

## 2. Check Environment

In [ ]:
import torch
import awq
import transformers

print(f"PyTorch version:      {torch.__version__}")
print(f"AutoAWQ version:      {awq.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")
print(f"CUDA version:         {torch.version.cuda}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_mem / 1e9:.1f} GB)")

## 3. Configuration

Set the model and quantization parameters:

In [ ]:
# ===== CONFIGURATION =====
MODEL_ID = "Qwen/Qwen3-4B-Instruct"      # Source model on HuggingFace
OUTPUT_DIR = "./qwen3-4b-instruct-awq"     # Where to save quantized model

# AWQ quantization parameters
QUANT_CONFIG = {
    "zero_point": True,      # Asymmetric quantization (recommended)
    "q_group_size": 128,     # Weights per quantization group
    "w_bit": 4,              # 4-bit quantization
    "version": "GEMM"        # GEMM kernel (best for batched inference)
}

# Calibration settings
CALIB_DATA = "wikitext"       # Calibration dataset

print("Configuration:")
print(f"  Model:        {MODEL_ID}")
print(f"  Output:       {OUTPUT_DIR}")
print(f"  Quant config: {QUANT_CONFIG}")
print(f"  Calib data:   {CALIB_DATA}")

## 4. Load the Model

Load the FP16 model and tokenizer. This will download the model from HuggingFace if not cached.

In [ ]:
import time
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

print("Loading model (this may take a few minutes for first download)...")
start = time.time()

model = AutoAWQForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

elapsed = time.time() - start
print(f"Model loaded in {elapsed:.1f}s")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

## 5. Check GPU Memory Before Quantization

In [ ]:
def print_gpu_memory():
    """Print GPU memory usage for all available GPUs."""
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"GPU {i}: {allocated:.1f} GB allocated / {reserved:.1f} GB reserved / {total:.1f} GB total")

print("GPU memory usage after loading FP16 model:")
print_gpu_memory()

## 6. Run AWQ Quantization

This is the main step! AWQ will:
1. Fetch calibration data (WikiText-2)
2. Collect activation statistics per channel
3. Find optimal scaling factors via grid search
4. Apply scaling and quantize weights to INT4

**Expected time: ~10-15 minutes on H100**

In [ ]:
print("Starting AWQ quantization...")
print(f"Config: {QUANT_CONFIG}")
print(f"Calibration data: {CALIB_DATA}")
print()

start = time.time()

model.quantize(
    tokenizer,
    quant_config=QUANT_CONFIG,
    calib_data=CALIB_DATA,
)

elapsed = time.time() - start
print(f"\nQuantization completed in {elapsed:.1f}s ({elapsed/60:.1f} minutes)")
print("\nGPU memory after quantization:")
print_gpu_memory()

## 7. Save the Quantized Model

In [ ]:
import os

print(f"Saving quantized model to {OUTPUT_DIR}...")

# Save model and tokenizer
model.save_quantized(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Print saved files
print("\nSaved files:")
total_size = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        fsize = os.path.getsize(fpath)
        total_size += fsize
        print(f"  {f:40s} {fsize/1e6:>8.1f} MB")
print(f"  {'TOTAL':40s} {total_size/1e9:>8.2f} GB")

## 8. Verify: Quick Inference Test

Let's load the quantized model and verify it produces coherent output.

In [ ]:
# Clear GPU memory from quantization
del model
torch.cuda.empty_cache()

# Load the quantized model
print("Loading quantized model for verification...")
quant_model = AutoAWQForCausalLM.from_quantized(
    OUTPUT_DIR,
    fuse_layers=True,
    device_map="auto"
)
quant_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

print("Quantized model loaded!")
print(f"\nGPU memory with quantized model:")
print_gpu_memory()

In [ ]:
# Test with a few prompts
test_prompts = [
    "What is the capital of France?",
    "Explain quantum computing in 2 sentences.",
    "Write a Python function to calculate factorial.",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = quant_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = quant_tokenizer(text, return_tensors="pt").to(quant_model.device)
    
    start = time.time()
    with torch.no_grad():
        outputs = quant_model.generate(
            **inputs, max_new_tokens=150, temperature=0.7, do_sample=True
        )
    gen_time = time.time() - start
    
    response = quant_tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    tokens_generated = outputs.shape[1] - inputs.input_ids.shape[1]
    
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print(f"[{tokens_generated} tokens in {gen_time:.1f}s = {tokens_generated/gen_time:.1f} tok/s]")

## 9. Compare Model Sizes

Let's see how much compression we achieved:

In [ ]:
import json

# Load and display quantization config
config_path = os.path.join(OUTPUT_DIR, "config.json")
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    if "quantization_config" in config:
        print("Quantization config saved in model:")
        print(json.dumps(config["quantization_config"], indent=2))

# Calculate compression stats
params_billions = 4.0  # Qwen3-4B
fp16_size_gb = params_billions * 2  # 2 bytes per param in FP16
awq_size_gb = total_size / 1e9
compression = fp16_size_gb / awq_size_gb

print(f"\nCompression Summary:")
print(f"  FP16 size (estimated): {fp16_size_gb:.1f} GB")
print(f"  AWQ INT4 size:         {awq_size_gb:.2f} GB")
print(f"  Compression ratio:     {compression:.2f}×")

## 10. (Optional) Upload to HuggingFace Hub

In [ ]:
# Uncomment and modify to upload
# from huggingface_hub import HfApi
# 
# REPO_ID = "your-username/Qwen3-4B-Instruct-AWQ"
# 
# api = HfApi()
# api.create_repo(REPO_ID, exist_ok=True)
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id=REPO_ID,
#     commit_message="Upload AWQ quantized Qwen3-4B-Instruct"
# )
# print(f"Model uploaded to https://huggingface.co/{REPO_ID}")

## ✅ Done!

Your quantized model is saved at the output directory. Next steps:

1. **Run inference** → See the [inference notebook](inference_awq.ipynb)
2. **Serve with vLLM** → `vllm serve ./qwen3-4b-instruct-awq --quantization awq`
3. **Upload to HuggingFace** → Uncomment the cell above